# Our run against AMW, chip by chip

Does our `detect.py` reproduce AMW's published quarterly detections? For every chip (the same
48 px square on both sides) we compare the raw detection, confidence > 0.4, with AMW's file of
the same quarter. Area is left out. AMW has no months, so our three months are compared with
AMW's quarter.

Works for any run: the 36-tile test region, one subregion, or the whole basin. Set `RUN` and
`QUARTERS`, then *Run All*. The join (section 1) reads every chip of the run once and saves a
small results folder, `<run>/amw_comparison/`; after that the notebook takes seconds.

In [ ]:
RUN = "tests_output/aca_full_2026-10-23"          # the whole basin, copied from Goethe
QUARTERS = ["2025Q1", "2025Q2"]
# RUN = "tests_output/test_region_2026-10-19"     # the quick check: 36 tiles in subregion 2
# QUARTERS = ["2026Q1", "2026Q2", "2026Q3"]
REDO = False            # True: join again although the results folder exists
CHECK_LAST_DAY = False  # True: section 6 asks Earth Engine for the last day's scenes (small requests)

import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "panel_pipeline").is_dir())
sys.path.insert(0, str(REPO / "amw_comparison"))

import json
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import compare

RUN = Path(RUN) if Path(RUN).is_absolute() else REPO / RUN
RESULTS = RUN / "amw_comparison"
pd.set_option("display.float_format", "{:,.4f}".format)

## 1. The join

`compare.compare_run` lines every chip of every subregion up with AMW's file by AMW's location
key, adds our three months, and keeps the chips someone detected. A border tile is in two
subregions' lists, so we scored it twice; like AMW, we keep the higher score. Skipped when the
results exist. A whole-basin quarter takes a few minutes and about 5 to 8 GB of memory.

In [ ]:
needed = [RESULTS / "amw_files.csv"] + [RESULTS / f"chips_{q}.parquet" for q in QUARTERS]
if REDO or not all(p.exists() for p in needed):
    compare.compare_run(RUN, QUARTERS)
else:
    print(f"results exist in {RESULTS}; set REDO = True to join again")
chips = {q: pd.read_parquet(RESULTS / f"chips_{q}.parquet") for q in QUARTERS
         if (RESULTS / f"chips_{q}.parquet").exists()}
QUARTERS = list(chips)          # quarters with no complete subregion have no results
for d in chips.values():
    d["outcome"] = compare.outcome(d)
print(f"quarters: {QUARTERS}")

## 2. The tables

**AMW** publishes one file per quarter, one row per chip it scored above 0.4: the chip's square
and its `confidence`. **Ours** is one row per chip per period, for every chip, detected or not,
with image quality next to the score (`n_obs` images, `n_clear` clear looks per pixel).
**Joined**: one row per chip, both scores side by side (`amw_confidence` is empty where AMW has
no row), and our months as `conf_<month>`.

In [ ]:
q = QUARTERS[-1]
display(gpd.read_file(compare.amw_file(q), rows=3))
parts = sorted(RUN.glob(f"detect/quarterly/period={q}/subregion=*/part_*.parquet"))
if parts:
    display(pd.read_parquet(parts[0]).head(3))
cols = [c for c in chips[q] if c not in ("tile", "empty_share", "border_copies", "border_spread")]
pd.concat([chips[q][chips[q]["outcome"] == o].head(2) for o in compare.OUTCOMES[:3]])[cols]

## 3. Inputs

Every task read must be complete with no failed tiles: a missing tile looks like "AMW detected it,
we didn't". AMW replaces its files when it re-runs, so the fingerprint says which version we
compared. *Should have matched*: AMW chips that fall between our chips; must be 0, or the grids
differ. *Chips sharing a key*: the same ground in two tiles of one subregion (where UTM zones
meet); like AMW, the higher score is kept.

In [ ]:
inputs = pd.read_csv(RESULTS / "inputs.csv")
scored = inputs[inputs["state"] != "missing"]
problems = scored[(scored["state"] != "complete") | (scored["failed"] > 0)]
print(f"{len(scored)} tasks scored ({len(inputs) - len(scored)} not scored in this run), "
      f"{len(problems)} incomplete or with failed tiles")
display(problems)
display(pd.read_csv(RESULTS / "amw_files.csv"))
joins = pd.read_csv(RESULTS / "joins.csv")
joins

## 4. Counts

Chips above 0.4. *Overlap* = chips both detected / chips either detected: 1 means the same set.

In [ ]:
whole = ["both", "only ours", "only AMW", "ours total", "AMW total"]
summary = pd.DataFrame({q: compare.agreement(d) for q, d in chips.items()}).T.drop(columns="neither")
summary[whole] = summary[whole].astype(int)
summary

In [ ]:
# per subregion (a border chip counts where its score was kept)
per_sub = pd.concat({q: d.groupby("subregion").apply(compare.agreement, include_groups=False)
                     for q, d in chips.items()}, names=["quarter"]).drop(columns="neither")
per_sub[whole] = per_sub[whole].astype(int)
per_sub

## 5. Scores

*Gap* = |our confidence − AMW's| over chips both detected. About 1e-7 means the same images went
in. Disagreements near 0.4 are small score differences that happen to cross the threshold.

In [ ]:
display(pd.DataFrame({q: compare.score_gap(d) for q, d in chips.items()}).T)
fig, axes = plt.subplots(1, len(QUARTERS), figsize=(4.5 * len(QUARTERS), 4), squeeze=False)
for ax, q in zip(axes[0], QUARTERS):
    d = chips[q]
    ax.scatter(d["amw_confidence"], d["confidence"], s=2, alpha=0.3)
    ax.plot([0, 1], [0, 1], "k--", lw=0.6)
    ax.axhline(0.4, color="grey", lw=0.6); ax.axvline(0.4, color="grey", lw=0.6)
    ax.set(title=q, xlabel="AMW confidence", ylabel="our confidence", xlim=(0, 1), ylim=(0, 1))
plt.tight_layout()
plt.show()
for q, d in chips.items():
    dis = d[d["outcome"].isin(["only ours", "only AMW"])]
    near = ((dis["confidence"] - 0.4).abs() <= 0.05).sum()
    print(f"{q}: {len(dis):,} disagreements, {near:,} with our score within 0.35-0.45")

In [ ]:
# image quality by outcome (medians): few clear looks make a quarter's image sensitive to one scene
pd.concat({q: d.groupby("outcome")[["confidence", "amw_confidence", "n_obs", "n_clear"]].median()
           for q, d in chips.items()}, names=["quarter"])

## 6. Why the scores differ: the last day

AMW's quarter stops one day early. Earth Engine excludes the end date, and AMW passes the
quarter's last day as the end, so 31 Mar, 30 Jun, 30 Sep and 31 Dec are never used. Ours ends on
the 1st of the next month, so every day is in. The dates are in AMW's file and in our task's
`_status.json`:

In [ ]:
rows = {}
for q in QUARTERS:
    name = compare.amw_file(q).open().read(400).split('"name": "')[1].split('"')[0]
    task = next(RUN.glob(f"detect/quarterly/period={q}/subregion=*/_status.json"), None)
    ours = json.loads(task.read_text()) if task else {}
    rows[q] = {"AMW start": name.split("_")[-2], "AMW end (excluded)": name.split("_")[-1],
               "our start": ours.get("start_date"), "our end (excluded)": ours.get("end_date_excluded")}
pd.DataFrame(rows).T

**Settled on Pará, 2026Q1** (20 Oct): AMW's own code, run with AMW's end date, reproduces AMW's
published file exactly (27 of 27 chips, identical scores); run with our end date, it reproduces
our `detect.py` scores exactly (56 of 56). Our code is AMW's code; only the end date differs.
We keep the last day: our quarters are whole calendar quarters, like our months.

If that is the whole difference here too, every chip scored differently lies under a scene taken
on the last day, and chips elsewhere are identical (a cloudy last-day scene changes nothing, so
identical chips under it are fine). Needs `CHECK_LAST_DAY = True`.

In [ ]:
outlines = {}
if not CHECK_LAST_DAY:
    print("skipped: set CHECK_LAST_DAY = True in the first cell (needs Earth Engine)")
else:
    import last_day
    last_day.login(RUN)
    for q in QUARTERS:
        scenes, outlines[q] = last_day.last_day_scenes(chips[q], q)
        print(f"{q}: {len(scenes)} scenes on {last_day.windows(q)[1]}, mean cloud {scenes['cloud_pct'].mean():.0f}%")
        display(last_day.under_last_day(chips[q], outlines[q]))

The map: grey where both scores are identical, coloured by the gap where they differ, red
triangles where only one side detects. With `CHECK_LAST_DAY`, the last day's scenes in blue.

In [ ]:
fig, axes = plt.subplots(1, len(QUARTERS), figsize=(6 * len(QUARTERS), 5), squeeze=False)
for ax, q in zip(axes[0], QUARTERS):
    d = chips[q].assign(gap=(chips[q]["confidence"] - chips[q]["amw_confidence"]).abs())
    both = d[d["outcome"] == "both"]
    same, differ = both[both["gap"] <= 1e-6], both[both["gap"] > 1e-6]
    ax.scatter(same["amw_key_x"], same["amw_key_y"], s=1, c="lightgrey", label=f"identical ({len(same):,})")
    sc = ax.scatter(differ["amw_key_x"], differ["amw_key_y"], s=2, c=differ["gap"], cmap="viridis",
                    vmin=0, vmax=0.05, label=f"gap > 0 ({len(differ):,})")
    for label, marker in (("only ours", "^"), ("only AMW", "v")):
        o = d[d["outcome"] == label]
        ax.scatter(o["amw_key_x"], o["amw_key_y"], s=12, marker=marker, facecolors="none",
                   edgecolors="red", lw=0.5, label=f"{label} ({len(o):,})")
    for poly in getattr(outlines.get(q), "geoms", [outlines.get(q)]):
        if poly is not None and not poly.is_empty:
            ax.plot(*poly.exterior.xy, color="tab:blue", lw=0.8)
    ax.set(title=q, xlabel="longitude", ylabel="latitude",
           xlim=(d["amw_key_x"].min() - 0.1, d["amw_key_x"].max() + 0.1),
           ylim=(d["amw_key_y"].min() - 0.1, d["amw_key_y"].max() + 0.1))
    ax.set_aspect("equal")
    ax.legend(fontsize=7, loc="best")
fig.colorbar(sc, ax=axes[0].tolist(), label="gap", shrink=0.7)
plt.show()

## 7. Border chips

A tile on the line between two subregions is in both subregions' lists, so it was scored twice,
on our side and on AMW's. *Border* = a location scored twice in this run (so only between two
subregions that are both scored). *Our two scores identical* checks that both copies saw the same
images; a gap to AMW at the border then has the same cause as elsewhere.

In [ ]:
rows = {}
for q, d in chips.items():
    border = d["border_copies"] > 1
    if not border.any():
        print(f"{q}: no location scored twice (border chips need both neighbouring subregions scored)")
    for name, part in (("border", d[border]), ("elsewhere", d[~border])):
        if part.empty:
            continue
        a, g = compare.agreement(part), compare.score_gap(part)
        rows[(q, name)] = {"chips": len(part), "both": a["both"], "only ours": a["only ours"],
                           "only AMW": a["only AMW"], "overlap": a["overlap (Jaccard)"],
                           "identical to AMW": g["identical (gap <= 1e-6)"],
                           "our two scores identical": (part["border_spread"] <= 1e-6).mean() if name == "border" else np.nan,
                           "largest gap between our two": part["border_spread"].max() if name == "border" else np.nan}
border_table = pd.DataFrame(rows).T
counts = ["chips", "both", "only ours", "only AMW"]
border_table[counts] = border_table[counts].astype(int)
border_table

## 8. Our months against AMW's quarter

How many of the quarter's three months detect a chip. A mine visible all quarter should be
detected in most months; clouds make single months miss. *Any month* is the union of the three.

In [ ]:
rows = {}
for q, d in chips.items():
    if d["months_scored"].max() == 0:
        print(f"{q}: no months scored, skipped")
        continue
    amw, ours, anym = d["amw_confidence"] > 0.4, d["confidence"] > 0.4, d["months_detected"] > 0
    rows[q] = {"months scored": int(d["months_scored"].max()),
               "AMW total": int(amw.sum()), "any month total": int(anym.sum()),
               "AMW and any month": int((amw & anym).sum()),
               "AMW, no month": int((amw & ~anym).sum()),
               "any month, not AMW": int((anym & ~amw).sum()),
               "overlap AMW / any month": (amw & anym).sum() / (amw | anym).sum(),
               "overlap our quarter / any month": (ours & anym).sum() / (ours | anym).sum()}
if rows:
    display(pd.DataFrame(rows).T)
    # AMW's detections by how many of our months detected them (0 to 3)
    display(pd.DataFrame({q: chips[q].loc[chips[q]["amw_confidence"] > 0.4, "months_detected"]
                             .value_counts().reindex(range(4), fill_value=0)
                          for q in rows}).rename_axis("months detected"))

## 9. Summary

In [ ]:
for q in QUARTERS:
    c, g = summary.loc[q], compare.score_gap(chips[q])
    print(f"{q}: ours {c['ours total']:,.0f}, AMW {c['AMW total']:,.0f}; both {c['both']:,.0f}, "
          f"only ours {c['only ours']:,.0f}, only AMW {c['only AMW']:,.0f} (overlap {c['overlap (Jaccard)']:.3f}); "
          f"scores identical for {g['identical (gap <= 1e-6)']:.0%}, largest gap {g['largest gap']:.3f}")